# Reweighing — ACSIncome (Illinois, 2018)

Test whether reweighing training observations reduces group disparities for Logistic Regression and Random Forest. We compare both models against the validation-set baselines; the held-out test set is reserved for final evaluation.

## Data and sample weights

Use the same 60/20/20 stratified split and model pipelines as the baseline experiment. Reweighing assigns each training observation a weight based on the frequency of its `SEX` group and income label. Weights are estimated **only from training data**.

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from fairlearn.metrics import false_positive_rate, true_positive_rate
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_acs_income
from src.experiment import prepare_data_splits
from src.models import create_logistic_regression, create_random_forest

In [10]:
X, y, A = load_acs_income()
X_train, X_val, X_test, y_train, y_val, y_test, A_train, A_val, A_test = (
    prepare_data_splits(X, y, A)
)

In [11]:
sample_weights = np.ones(len(y_train), dtype=float)

for group in np.unique(A_train):
    group_frequency = np.mean(A_train == group)
    for label in np.unique(y_train):
        mask = (A_train == group) & (y_train == label)
        joint_frequency = np.mean(mask)
        if joint_frequency == 0:
            continue
        label_frequency = np.mean(y_train == label)
        sample_weights[mask] = (
            group_frequency * label_frequency / joint_frequency
        )

In [12]:
models = {
    "Logistic Regression": create_logistic_regression(),
    "Random Forest": create_random_forest(),
}
for model in models.values():
    model.fit(X_train, y_train, classifier__sample_weight=sample_weights)

## Validation results

Evaluate predictive performance and absolute group gaps on the validation split. EOD is the absolute difference in true positive rates; DPD is the absolute difference in positive prediction rates; FPR Gap is the absolute difference in false positive rates.

In [13]:
def evaluate_reweighing(name, model):
    predictions = model.predict(X_val)
    male = np.asarray(A_val) == 1
    female = np.asarray(A_val) == 2

    tpr_gap = (
        true_positive_rate(y_val[male], predictions[male])
        - true_positive_rate(y_val[female], predictions[female])
    )
    fpr_gap = (
        false_positive_rate(y_val[male], predictions[male])
        - false_positive_rate(y_val[female], predictions[female])
    )
    positive_rate_gap = np.mean(predictions[male]) - np.mean(predictions[female])

    return {
        "Model": name,
        "Strategy": "Reweighing",
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
        "EOD": abs(tpr_gap),
        "DPD": abs(positive_rate_gap),
        "FPR Gap": abs(fpr_gap),
    }

In [14]:
reweighing_results = pd.DataFrame([
    evaluate_reweighing(name, model) for name, model in models.items()
])
display(reweighing_results.round(4))

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Reweighing,0.7994,0.7870,0.0349,0.0923,0.0075
1,Random Forest,Reweighing,0.7839,0.7666,0.0081,0.1027,0.0203


## Comparison with baseline

Compare the mitigation strategy with the baseline results from `02_baseline_clean.ipynb`. Lower fairness gaps indicate smaller disparities under the respective metric, but a decrease in one gap does not guarantee improvement in others.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
baseline_path = results_dir / "baseline_comparison.csv"
if not baseline_path.exists():
    raise FileNotFoundError(
        "Run 02_baseline_clean.ipynb first to generate results/baseline_comparison.csv"
    )

baseline_results = pd.read_csv(baseline_path)
comparison = pd.concat([baseline_results, reweighing_results], ignore_index=True)
comparison = comparison.sort_values(["Model", "Strategy"]).reset_index(drop=True)
display(comparison.round(4))

reweighing_results.to_csv(results_dir / "reweighing_comparison.csv", index=False)
comparison.to_csv(results_dir / "baseline_vs_reweighing.csv", index=False)

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Baseline,0.8020,0.7901,0.0215,0.1405,0.0515
1,Logistic Regression,Reweighing,0.7994,0.7870,0.0349,0.0923,0.0075
2,Random Forest,Baseline,0.7868,0.7706,0.0294,0.1305,0.0413
3,Random Forest,Reweighing,0.7839,0.7666,0.0081,0.1027,0.0203


## Interpretation and limitations

Reweighing changes the influence of training observations without using `SEX` as a predictor. Its effect may differ between classifiers and across fairness metrics. These are validation-set comparisons, not final test estimates; statistical uncertainty and test-set performance are evaluated later. The group gaps describe observed differences and do not establish discriminatory causation.